# Day 53：LLaMA-Factory 微调实战与 Loss 曲线诊断（Jupyter Notebook 实战手册）

> **所属阶段**：第 5 阶段 · 第五阶段：模型微调与部署 ｜ **第 8 周**：模型微调、本地推理部署与安全合规  
> **GitLab 工单**：`STARMIND-053` ｜ **开发分支**：`feat/day53-llamafactory-yaml-loss`  
> **配套 HTML5 课件**：`courseware/day53/index.html` ｜ **独立版本快照**：`versions/day53/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天正式开炉炼丹——LLaMA-Factory 微调实战。为啥国内企业 90% 都用 LLaMA-Factory？因为它是北航开源的宝藏框架，Qwen2.5、DeepSeek、GLM、Llama 全支持，既有 WebUI 点点鼠标，又能导出标准 YAML 走命令行 CI 自动化。但我要提前打个预防针：很多人把 YAML 跑起来就不管了，看到 train_loss 掉到 0.05 还开心得发朋友圈，结果 eval_loss 早就飙上天了——那是严重过拟合！今天咱手写两个硬核工具：一个是防呆的 LLaMA-Factory YAML 配置生成器，一个是能自动揪出过拟合、欠拟合和梯度爆炸的 Loss 诊断引擎。

### 承上启下
- **昨日基线**：昨天我们产出并质检了 500 条高纯度星辰客服微调语料（450 训练 + 50 验证），已经做好了入炉准备。
- **明日预告**：明天（Day 54）我们解决微调完之后的事：用自动化评测集对比微调前后效果，并学习如何将 LoRA 权重无损合并回底座，以及 GPTQ/AWQ/GGUF 量化部署原理。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day53'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-053）

团队准备使用 LLaMA-Factory 对 Qwen2.5-7B-Instruct 执行 LoRA 微调。此前新人手工修改 YAML 配置常因 template 选错（如给 Qwen2.5 配了 llama3 模板）、learning_rate 过高（1e-3 导致乱码）或有效 batch_size 失衡导致训练失败。

**今日迭代目标：**
- 实现防呆的 LLaMA-Factory SFT LoRA YAML 生成器，自动匹配模型模板、计算全局有效 Batch Size 与总训练步数。
- 实现训练日志 Loss 曲线诊断器，自动识别正常收敛、过拟合（eval_loss 拐头上升）、欠拟合与学习率过大（NaN/震荡），并推荐最优 checkpoint 步数。

**验收标准：**
- 针对 Qwen2.5-7B 自动锁定 `template: qwen`，强制检查 LoRA 学习率在 `[1e-5, 5e-4]` 合理区间。
- Loss 诊断器能准确区分健康收敛、过拟合、欠拟合与梯度发散四种曲线模式，并给出可执行的调参建议。
- 纯 CPU 环境下 pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 国内环境安装与加速必备环境变量

在国内机器或 AutoDL 上运行 LLaMA-Factory，直接连 HuggingFace 常超时。只需在启动前设置 `export USE_MODELSCOPE_HUB=1`（魔搭镜像）或 `export HF_ENDPOINT=https://hf-mirror.com`，模型权重下载速度可直接拉满千兆带宽。

> **⚠️ 老陈避坑**：坑点：忘记把 `bf16: true` 配在支持 BF16 的 30/40 系及 A100/A800 显卡上而误用 `fp16: true`，Qwen2.5 在 FP16 下比 BF16 更容易出现梯度溢出 NaN。

### 2. 2. lora_rank 与 lora_alpha 的配比规律

常规指令微调推荐 `lora_rank: 16`、`lora_alpha: 32`（即 alpha = 2 × rank，缩放因子 α/r = 2.0）。如果任务非常简单（仅改语气），rank=8 即可；若涉及复杂领域推理或代码生成，可提至 rank=32 或 64，同时配合 `lora_dropout: 0.05` 抑制过拟合。

> **⚠️ 老陈避坑**：坑点：把 rank 盲目拉到 256，不仅显存暴涨，在小数据集上反而更容易死记硬背。

### 3. 3. 什么时候该早停（Early Stopping）？

微调不是训得越久越好！当 `eval_loss` 达到全局最低点后连续 2~3 个评估周期不再下降甚至反升，说明模型已经开始死背训练集里的噪声和标点习惯。此时必须果断回滚加载 `eval_loss` 最低的那个 checkpoint。

> **⚠️ 老陈避坑**：坑点：只在训练结束保存最后一个权重，没设 `save_steps` 和 `load_best_model_at_end: true`，导致最后拿到的是过拟合最严重的版本。


## 💻 实操步骤 1：步骤一：实现防呆的 LLaMA-Factory LoRA 微调 YAML 配置生成器

- **目标文件**：`versions/day53/src/day53_llamafactory_config.py`
- **老陈代码旁白**：每次微调都手抄 YAML 太容易踩坑了。我们写一个带强校验的配置生成器，自动映射 chat template、校验学习率范围，并帮你算好总训练步数和评估频率。


In [ ]:
"""
Day 53 - 模块一：LLaMA-Factory LoRA 微调 YAML 配置生成器（带防呆校验）
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Dict

MODEL_TEMPLATE_MAP: Dict[str, str] = {
    "Qwen/Qwen2.5-7B-Instruct": "qwen",
    "Qwen/Qwen2.5-14B-Instruct": "qwen",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B": "deepseek3",
    "meta-llama/Meta-Llama-3.1-8B-Instruct": "llama3",
}


@dataclass
class LoRATrainPlan:
    """微调训练超参计划"""
    model_name_or_path: str = "Qwen/Qwen2.5-7B-Instruct"
    dataset_name: str = "starmind_cs_alpaca"
    dataset_size: int = 450
    output_dir: str = "saves/qwen2.5-7b/lora/starmind_cs_v1"
    lora_rank: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    learning_rate: float = 1e-4
    num_train_epochs: float = 4.0
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_gpus: int = 1
    cutoff_len: int = 2048
    use_qlora_4bit: bool = False


def validate_and_render_yaml(plan: LoRATrainPlan) -> str:
    """校验超参合理性并渲染标准 LLaMA-Factory YAML 配置"""
    if plan.model_name_or_path not in MODEL_TEMPLATE_MAP:
        raise ValueError(f"未注册的模型模板映射: {plan.model_name_or_path}")
    if not (1e-5 <= plan.learning_rate <= 5e-4):
        raise ValueError(
            f"LoRA 学习率 {plan.learning_rate} 超出安全区间 [1e-5, 5e-4]，易导致欠拟合或梯度爆炸"
        )
    if plan.lora_rank <= 0 or plan.lora_alpha < plan.lora_rank:
        raise ValueError("lora_rank 必须为正整数，且建议 lora_alpha >= lora_rank")

    template = MODEL_TEMPLATE_MAP[plan.model_name_or_path]
    ebs = plan.per_device_train_batch_size * plan.gradient_accumulation_steps * plan.num_gpus
    steps_per_epoch = max(1, math.ceil(plan.dataset_size / ebs))
    total_steps = int(steps_per_epoch * plan.num_train_epochs)
    eval_steps = max(10, total_steps // 6)

    quant_block = "quantization_bit: 4\nquantization_method: bitsandbytes\n" if plan.use_qlora_4bit else ""

    yaml_text = f"""### 自动生成：星辰智脑 LLaMA-Factory LoRA 微调配置 ###
### 预估 Effective Batch Size = {ebs}, 总步数 ≈ {total_steps} steps ###

### model
model_name_or_path: {plan.model_name_or_path}
{quant_block}trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: {plan.lora_rank}
lora_alpha: {plan.lora_alpha}
lora_dropout: {plan.lora_dropout}
lora_target: all

### dataset
dataset: {plan.dataset_name}
template: {template}
cutoff_len: {plan.cutoff_len}
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: {plan.output_dir}
logging_steps: 5
save_steps: {eval_steps}
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: {plan.per_device_train_batch_size}
gradient_accumulation_steps: {plan.gradient_accumulation_steps}
learning_rate: {plan.learning_rate:.1e}
num_train_epochs: {plan.num_train_epochs}
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
gradient_checkpointing: true

### eval
val_size: 0.1
per_device_eval_batch_size: 2
eval_strategy: steps
eval_steps: {eval_steps}
load_best_model_at_end: true
"""
    return yaml_text


if __name__ == "__main__":
    cfg = validate_and_render_yaml(LoRATrainPlan())
    print(cfg)


### 🔍 步骤 1 代码深度解析

1. 自动校验 `learning_rate` 和 `template`，从源头掐断 80% 的低级配置失误。
2. 自动计算 `Effective Batch Size`、`total_steps` 与合理的 `eval_steps`（约每训练 1/6 进度验证一次），并默认开启 `load_best_model_at_end: true` 防止拿到过拟合尾部权重。

**预期运行输出参考：**
```text
### 自动生成：星辰智脑 LLaMA-Factory LoRA 微调配置 ###
### 预估 Effective Batch Size = 8, 总步数 ≈ 228 steps ###
...
```


## 💻 实操步骤 2：步骤二：实现训练 Loss 曲线自动诊断与早停推荐引擎

- **目标文件**：`versions/day53/src/day53_loss_diagnostics.py`
- **老陈代码旁白**：训练跑起来后，光靠肉眼盯屏幕太低效。我们写一个 Loss 诊断器，吃进每一步的 train_loss 和 eval_loss，直接判断当前曲线属于健康、过拟合、欠拟合还是梯度爆炸，并告诉你该回滚哪个 checkpoint。


In [ ]:
"""
Day 53 - 模块二：微调 Loss 曲线模拟与过拟合/欠拟合自动诊断引擎
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple


@dataclass
class LossPoint:
    """单个评估检查点的训练/验证损失"""
    step: int
    train_loss: float
    eval_loss: float


@dataclass
class DiagnosisResult:
    """Loss 曲线诊断报告"""
    status: str                  # "healthy" | "overfitting" | "underfitting" | "diverging"
    best_step: Optional[int]
    best_eval_loss: Optional[float]
    overfit_gap: float
    advice: str


def diagnose_loss_curve(points: List[LossPoint]) -> DiagnosisResult:
    """分析训练与验证 Loss 序列，输出健康状态、最佳早停点与调参建议"""
    if len(points) < 3:
        raise ValueError("至少需要 3 个评估点才能进行趋势诊断")

    # 1. 检测梯度爆炸 / NaN / 剧烈反弹
    for p in points:
        if math.isnan(p.train_loss) or math.isnan(p.eval_loss) or p.train_loss > 10.0:
            return DiagnosisResult(
                status="diverging",
                best_step=None,
                best_eval_loss=None,
                overfit_gap=0.0,
                advice="检测到 Loss 出现 NaN 或剧烈发散！请将 learning_rate 减半(如 5e-5)、开启 bf16 并检查脏数据。",
            )

    if points[-1].train_loss > points[0].train_loss * 1.15:
        return DiagnosisResult(
            status="diverging",
            best_step=points[0].step,
            best_eval_loss=points[0].eval_loss,
            overfit_gap=round(points[-1].eval_loss - points[-1].train_loss, 4),
            advice="训练 Loss 不降反升，学习率过大或数据格式模板错位，请检查 template 与 lr。",
        )

    best_pt = min(points, key=lambda x: x.eval_loss)
    last_pt = points[-1]
    gap = round(last_pt.eval_loss - last_pt.train_loss, 4)

    # 2. 检测过拟合：eval_loss 在最低点之后明显反弹(>8%)且 train_loss 仍在下降
    rebound_ratio = (last_pt.eval_loss - best_pt.eval_loss) / max(best_pt.eval_loss, 1e-6)
    if best_pt.step < last_pt.step and rebound_ratio >= 0.08 and last_pt.train_loss < best_pt.train_loss:
        return DiagnosisResult(
            status="overfitting",
            best_step=best_pt.step,
            best_eval_loss=round(best_pt.eval_loss, 4),
            overfit_gap=gap,
            advice=(
                f"检测到过拟合（验证集 Loss 较最低点反弹 {rebound_ratio*100:.1f}%）。"
                f"建议直接回滚使用 checkpoint-{best_pt.step}，下轮减少 epoch 或增大 lora_dropout 至 0.1。"
            ),
        )

    # 3. 检测欠拟合：整体降幅过小且最终 Loss 仍偏高
    train_drop = (points[0].train_loss - last_pt.train_loss) / max(points[0].train_loss, 1e-6)
    if train_drop < 0.15 and last_pt.train_loss > 1.8:
        return DiagnosisResult(
            status="underfitting",
            best_step=best_pt.step,
            best_eval_loss=round(best_pt.eval_loss, 4),
            overfit_gap=gap,
            advice="检测到欠拟合（训练 Loss 降幅不足 15% 且仍高于 1.8）。建议调高学习率至 1e-4、增大 lora_rank 或增加 epoch。",
        )

    return DiagnosisResult(
        status="healthy",
        best_step=best_pt.step,
        best_eval_loss=round(best_pt.eval_loss, 4),
        overfit_gap=gap,
        advice=f"训练收敛健康！建议使用最优权重 checkpoint-{best_pt.step} 进入下一阶段自动评测。",
    )


if __name__ == "__main__":
    overfit_curve = [
        LossPoint(40, 2.15, 2.20),
        LossPoint(80, 1.35, 1.48),
        LossPoint(120, 0.85, 1.12),
        LossPoint(160, 0.42, 1.24),
        LossPoint(200, 0.18, 1.41),
    ]
    print("过拟合场景诊断:", diagnose_loss_curve(overfit_curve))


### 🔍 步骤 2 代码深度解析

通过对比 `best_pt.eval_loss` 与 `last_pt.eval_loss` 的反弹比例，精准识别典型的'剪刀差'过拟合，并直接给出需要回滚的 `checkpoint-{best_step}`，避免把整轮训练成果作废。

**预期运行输出参考：**
```text
过拟合场景诊断: DiagnosisResult(status='overfitting', best_step=120, best_eval_loss=1.12, overfit_gap=1.23, advice='检测到过拟合（验证集 Loss 较最低点反弹 25.9%）。建议直接回滚使用 checkpoint-120，下轮减少 epoch 或增大 lora_dropout 至 0.1。')
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day53/tests/test_day53.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 53 单元测试：LLaMA-Factory YAML 生成与 Loss 曲线诊断"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import pytest
from src.day53_llamafactory_config import LoRATrainPlan, validate_and_render_yaml
from src.day53_loss_diagnostics import LossPoint, diagnose_loss_curve


def test_yaml_generation_contains_qwen_template_and_qlora():
    yaml_str = validate_and_render_yaml(LoRATrainPlan(use_qlora_4bit=True))
    assert "template: qwen" in yaml_str
    assert "quantization_bit: 4" in yaml_str
    assert "lora_target: all" in yaml_str


def test_yaml_rejects_dangerous_learning_rate():
    with pytest.raises(ValueError):
        validate_and_render_yaml(LoRATrainPlan(learning_rate=5e-3))


def test_diagnose_healthy_curve():
    pts = [LossPoint(30, 2.2, 2.3), LossPoint(60, 1.4, 1.5), LossPoint(90, 0.9, 1.05)]
    res = diagnose_loss_curve(pts)
    assert res.status == "healthy"
    assert res.best_step == 90


def test_diagnose_overfitting_finds_best_checkpoint():
    pts = [
        LossPoint(40, 2.0, 2.1),
        LossPoint(80, 1.1, 1.25),
        LossPoint(120, 0.6, 1.10),
        LossPoint(160, 0.25, 1.35),
    ]
    res = diagnose_loss_curve(pts)
    assert res.status == "overfitting"
    assert res.best_step == 120


def test_diagnose_diverging_nan():
    pts = [LossPoint(10, 2.0, 2.1), LossPoint(20, float("nan"), 2.5), LossPoint(30, 5.0, 6.0)]
    res = diagnose_loss_curve(pts)
    assert res.status == "diverging"

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为 YAML 生成器新增 DeepSpeed ZeRO-2 选项

**题目要求：**
扩展 `LoRATrainPlan`，当 `num_gpus >= 2` 时自动在渲染的 YAML 中追加 `deepspeed: examples/deepspeed/ds_z2_config.json` 配置项。

**思路提示：** 判断 `plan.num_gpus >= 2` 拼接一行字符串即可。


In [ ]:
from src.day53_llamafactory_config import LoRATrainPlan, validate_and_render_yaml


def render_with_deepspeed(plan: LoRATrainPlan) -> str:
    base_yaml = validate_and_render_yaml(plan)
    if plan.num_gpus >= 2:
        base_yaml += "deepspeed: examples/deepspeed/ds_z2_config.json\n"
    return base_yaml


out = render_with_deepspeed(LoRATrainPlan(num_gpus=2))
assert "ds_z2_config.json" in out
print("双卡 DeepSpeed 配置追加成功！")


**答案解析与评分标准：**

多卡训练配合 ZeRO-2 能显著降低每卡优化器与梯度开销，同时通信成本远低于 ZeRO-3，是 2~4 卡微调 7B/14B 模型的首选。


### 作业 2（企业实战）：企业实战：解析真实 LLaMA-Factory 的 trainer_log.jsonl 并诊断

**题目要求：**
编写函数 `parse_and_diagnose_jsonl(jsonl_text: str)`，从类似 `{"current_steps": 40, "loss": 1.8, "eval_loss": 1.9}` 的多行 JSONL 日志中提取 `LossPoint` 列表并调用 `diagnose_loss_curve`。

**思路提示：** 逐行 `json.loads`，过滤同时包含 `loss` 与 `eval_loss` 的记录（或将相邻 step 的 train/eval 合并）。


In [ ]:
import json
from src.day53_loss_diagnostics import LossPoint, diagnose_loss_curve


def parse_and_diagnose_jsonl(jsonl_text: str):
    points = []
    last_train = None
    for line in jsonl_text.strip().splitlines():
        row = json.loads(line)
        if "loss" in row:
            last_train = float(row["loss"])
        if "eval_loss" in row and last_train is not None:
            points.append(LossPoint(int(row["current_steps"]), last_train, float(row["eval_loss"])))
    return diagnose_loss_curve(points)


sample_log = """
{"current_steps": 40, "loss": 2.1}
{"current_steps": 40, "eval_loss": 2.15}
{"current_steps": 80, "loss": 1.2}
{"current_steps": 80, "eval_loss": 1.28}
{"current_steps": 120, "loss": 0.75}
{"current_steps": 120, "eval_loss": 0.92}
"""
print(parse_and_diagnose_jsonl(sample_log))


**答案解析与评分标准：**

LLaMA-Factory 的真实日志里 `loss` 和 `eval_loss` 往往分属相邻两行 JSON，用 `last_train` 缓存对齐是最稳健的解析方式。


### 作业 3（架构进阶）：架构进阶：实现 Cosine Learning Rate Scheduler 模拟曲线

**题目要求：**
手写带线性 Warmup 的余弦退火学习率函数 `cosine_lr(step, total_steps, base_lr, warmup_ratio=0.1)`，打印第 0、10%、50%、100% 步数时的学习率。

**思路提示：** 当 `step < warmup_steps` 时线性上升 `base_lr * step / warmup_steps`；之后按 `0.5 * base_lr * (1 + cos(pi * progress))` 平滑衰减至 0。


In [ ]:
import math


def cosine_lr(step: int, total_steps: int, base_lr: float, warmup_ratio: float = 0.1) -> float:
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    if step < warmup_steps:
        return base_lr * (step / warmup_steps)
    progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return 0.5 * base_lr * (1.0 + math.cos(math.pi * progress))


for s in (0, 10, 20, 100, 200):
    print(f"step={s:3d} -> lr={cosine_lr(s, 200, 1e-4):.6e}")


**答案解析与评分标准：**

Warmup 防止训练初期随机初始化的 LoRA 矩阵产生剧烈梯度震荡破坏底座，后期余弦退火帮助模型精细落入局部最优点。


---

## 🎯 老陈收尾复盘

漂亮！以后在星辰数科做微调，绝不允许手敲 50 行 YAML 碰运气，全部用今天这个生成器校验超参合法性后再提交训练；训完日志直接丢进 Loss 诊断器，健康、过拟合还是学习率炸了，一眼出结论。明天咱搞模型评测、权重合并和量化！
